<a href="https://colab.research.google.com/github/Mogi21/pytorch-deep-learning-labs/blob/main/02_pytorch_regression" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("[ Preparing Environment & Locating Data ]")

# 1. Dynamically find the exact folder containing the CSVs
target_dir = ''
for root, dirs, files in os.walk('/content'):
    if 'data_train.csv' in files:
        target_dir = root
        break

if not target_dir:
    raise FileNotFoundError("Could not find 'data_train.csv'. Please ensure data is uploaded and unzipped!")

print(f"Data successfully located in: {target_dir}")

# 2. Load data using the dynamic path
X_full_df = pd.read_csv(os.path.join(target_dir, 'data_train.csv'), index_col=0)
y_full_df = pd.read_csv(os.path.join(target_dir, 'labels_train.csv'), index_col=0)
X_test_df = pd.read_csv(os.path.join(target_dir, 'data_test.csv'), index_col=0)

# 3. Train/Validation Split (80/20)
X_train_df, X_valid_df, y_train_df, y_valid_df = train_test_split(
    X_full_df, y_full_df, test_size=0.2, random_state=42
)

# 4. Standardize features (Critical for PyTorch Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_df)
X_valid_scaled = scaler.transform(X_valid_df)
X_test_scaled = scaler.transform(X_test_df)

# 5. Custom PyTorch Dataset
class CaliforniaHousingDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32).view(-1, 1) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# 6. Create DataLoaders
BATCH_SIZE = 64
loader_train = DataLoader(CaliforniaHousingDataset(X_train_scaled, y_train_df), batch_size=BATCH_SIZE, shuffle=True)
loader_valid = DataLoader(CaliforniaHousingDataset(X_valid_scaled, y_valid_df), batch_size=BATCH_SIZE, shuffle=False)
loader_test = DataLoader(CaliforniaHousingDataset(X_test_scaled), batch_size=BATCH_SIZE, shuffle=False)

print(f"Ready! Training samples: {len(X_train_scaled)} | Features: {X_train_scaled.shape[1]}")

# ==========================================
# --- Architecture & Training Definition ---
# ==========================================

class RegressionNetwork(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layer = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.layer(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_regression(model, epochs=50, lr=0.01):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_batch, y_batch in loader_train:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        valid_losses = []
        with torch.no_grad():
            for X_batch, y_batch in loader_valid:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                valid_losses.append(loss_fn(model(X_batch), y_batch).item())

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Train MSE: {np.mean(train_losses):.4f} | Valid MSE: {np.mean(valid_losses):.4f}")

# ==========================================
# --- Execution & Prediction ---
# ==========================================

# Initialize model
model = RegressionNetwork(input_dim=X_train_scaled.shape[1]).to(device)
print(f"Training on {device}...")

# Train model
train_regression(model, epochs=50, lr=0.01)

# Generate Predictions
model.eval()
predictions = []

with torch.no_grad():
    for X_batch in loader_test:
        y_pred = model(X_batch.to(device))
        predictions.extend(y_pred.cpu().numpy())

# Save to CSV
pd.DataFrame(np.array(predictions).flatten(), columns=['predictions']).to_csv('submission.csv')
print("Predictions successfully saved to 'submission.csv'")

[ Preparing Environment & Locating Data ]
Data successfully located in: /content/data/task5
Ready! Training samples: 13209 | Features: 8
Training on cpu...
Epoch 01/50 | Train MSE: 1.3603 | Valid MSE: 2.8551
Epoch 10/50 | Train MSE: 140503.8144 | Valid MSE: 20023770.7465
Epoch 20/50 | Train MSE: 17645066282503.4219 | Valid MSE: 1399716554661257.7500
Epoch 30/50 | Train MSE: 1052542579711128240128.0000 | Valid MSE: 95463921141786091716608.0000
Epoch 40/50 | Train MSE: 51536008225950875521088749568.0000 | Valid MSE: 6563538222010458776244643168256.0000
Epoch 50/50 | Train MSE: inf | Valid MSE: inf
Predictions successfully saved to 'submission.csv'


In [1]:
#!unzip data.zip

Archive:  data.zip
  inflating: data/task5/data_train.csv  
  inflating: data/task5/labels_train.csv  
  inflating: data/task5/data_test.csv  
  inflating: data/task5/description.md  
